# TF‑IDF + XGBoost (Multiclass) — Improvement Playbook

This notebook helps you:
1. **Train** a baseline TF‑IDF + XGBoost multiclass model
2. **Save & load** the model for inference
3. **Evaluate** beyond accuracy (macro/weighted F1, log loss, top‑k, confusion matrix)
4. **Tackle imbalance** (class‑weighted training, oversampling)
5. **Reduce dimensionality** (chi2 feature selection, TruncatedSVD)
6. **Compare variants** in a single results table

**How to use**
- If you have your own dataset, fill in the **Your Dataset** cell.
- Otherwise, run the **20 Newsgroups demo** path.
- Execute cells top‑to‑bottom.

⚠️ Charts use matplotlib only (no seaborn).

In [ ]:
#!pip -q install xgboost imbalanced-learn scikit-learn pandas numpy matplotlib scipy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix, f1_score, log_loss, top_k_accuracy_score
from sklearn.preprocessing import LabelEncoder, Normalizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import joblib
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')
RANDOM_STATE = 42

## Load Data
### Option A) Your Dataset (CSV)
Provide a CSV with a **text** column and a **label** column. Adjust `TEXT_COL` and `LABEL_COL`.

In [ ]:
# === Your Dataset ===
USE_OWN_DATA = False  # Set to True to use your CSV
CSV_PATH = "/content/your_data.csv"  # Upload to Colab and set the path
TEXT_COL = "text"
LABEL_COL = "label"

if USE_OWN_DATA:
    df = pd.read_csv(CSV_PATH)
    df = df[[TEXT_COL, LABEL_COL]].dropna()
    texts = df[TEXT_COL].astype(str).tolist()
    labels = df[LABEL_COL].astype(str).tolist()
else:
    texts, labels = None, None
len(texts) if texts is not None else None

### Option B) Demo Dataset — 20 Newsgroups
This will download a public dataset and use its 20 categories as labels.

In [ ]:
if texts is None:
    from sklearn.datasets import fetch_20newsgroups
    newsgroups = fetch_20newsgroups(subset='all', remove=('headers','quotes','footers'))
    texts = [t if isinstance(t, str) else str(t) for t in newsgroups.data]
    labels = [newsgroups.target_names[i] for i in newsgroups.target]

print(f"Dataset size: {len(texts)}")
print("Sample label counts (top 10):")
from collections import Counter
print(Counter(labels).most_common(10))

## Train/Test Split (Stratified)
Stratification preserves class proportions in train and test.

In [ ]:
le = LabelEncoder()
y = le.fit_transform(labels)
X_train_texts, X_test_texts, y_train, y_test = train_test_split(
    texts, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
num_classes = len(le.classes_)
print(f"Train size: {len(X_train_texts)}, Test size: {len(X_test_texts)}, Classes: {num_classes}")

## Helper: Metrics & Reporting
We report **macro/weighted F1**, **log loss**, **top‑k accuracy**, and a confusion matrix.

In [ ]:
def evaluate_and_report(y_true, y_pred, y_proba, target_names):
    report = classification_report(y_true, y_pred, target_names=target_names, digits=4, zero_division=0)
    macro_f1 = f1_score(y_true, y_pred, average='macro')
    weighted_f1 = f1_score(y_true, y_pred, average='weighted')
    ll = log_loss(y_true, y_proba)
    top3 = top_k_accuracy_score(y_true, y_proba, k=min(3, y_proba.shape[1]))
    print(report)
    print(f"Macro-F1: {macro_f1:.4f}\nWeighted-F1: {weighted_f1:.4f}\nLogLoss: {ll:.4f}\nTop-3 Acc: {top3:.4f}")
    cm = confusion_matrix(y_true, y_pred)
    plt.figure()
    plt.imshow(cm)
    plt.title("Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.colorbar()
    plt.show()
    return {"macro_f1": macro_f1, "weighted_f1": weighted_f1, "log_loss": ll, "top3_acc": top3}

results = []  # to collect variant metrics

## Variant A — **Baseline**: TF‑IDF → XGBoost
We cap `max_features` to 50k for speed; adjust as needed. You can raise to 200k+ if memory allows.

In [ ]:
tfidf = TfidfVectorizer(stop_words='english', max_df=0.7, min_df=2, max_features=50000)
X_train = tfidf.fit_transform(X_train_texts)
X_test = tfidf.transform(X_test_texts)

clf_base = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=num_classes,
    n_estimators=400,
    learning_rate=0.15,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss',
    tree_method='hist',
    random_state=RANDOM_STATE
)
clf_base.fit(X_train, y_train)
proba_base = clf_base.predict_proba(X_test)
pred_base = np.argmax(proba_base, axis=1)
m_base = evaluate_and_report(y_test, pred_base, proba_base, list(le.classes_))
results.append({"variant": "Baseline TFIDF+XGB", **m_base})

### Save & Load Demo
We save both the **sklearn pipeline objects** and the **native XGBoost booster**.

In [ ]:
joblib.dump(tfidf, "tfidf_vectorizer.joblib")
joblib.dump(clf_base, "xgb_model.joblib")
clf_base.get_booster().save_model("xgb_model.json")
print("Saved: tfidf_vectorizer.joblib, xgb_model.joblib, xgb_model.json")

# Load and predict on a few samples
tfidf_l = joblib.load("tfidf_vectorizer.joblib")
clf_l = joblib.load("xgb_model.joblib")
sample_texts = X_test_texts[:5]
sample_X = tfidf_l.transform(sample_texts)
sample_pred = clf_l.predict(sample_X)
for i, p in enumerate(sample_pred):
    print(i, "→", le.classes_[p])

## Variant B — **Class-Weighted Training** (per‑example weights)
For multiclass imbalance, we compute inverse‑frequency class weights and pass them as `sample_weight` to `fit()`.

In [ ]:
classes = np.unique(y_train)
cw = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_to_w = {c: w for c, w in zip(classes, cw)}
w_train = np.array([class_to_w[c] for c in y_train])

clf_cw = xgb.XGBClassifier(
    objective='multi:softprob', num_class=num_classes, n_estimators=400,
    learning_rate=0.15, max_depth=8, subsample=0.8, colsample_bytree=0.8,
    eval_metric='mlogloss', tree_method='hist', random_state=RANDOM_STATE
)
clf_cw.fit(X_train, y_train, sample_weight=w_train)
proba_cw = clf_cw.predict_proba(X_test)
pred_cw = np.argmax(proba_cw, axis=1)
m_cw = evaluate_and_report(y_test, pred_cw, proba_cw, list(le.classes_))
results.append({"variant": "Class‑Weighted", **m_cw})

## Variant C — **Random Oversampling** (Imbalanced‑learn)
We perform oversampling **after** TF‑IDF to balance classes, then train XGBoost on the oversampled data.

In [ ]:
ros = RandomOverSampler(random_state=RANDOM_STATE)
X_ros, y_ros = ros.fit_resample(X_train, y_train)
clf_ros = xgb.XGBClassifier(
    objective='multi:softprob', num_class=num_classes, n_estimators=400,
    learning_rate=0.15, max_depth=8, subsample=0.8, colsample_bytree=0.8,
    eval_metric='mlogloss', tree_method='hist', random_state=RANDOM_STATE
)
clf_ros.fit(X_ros, y_ros)
proba_ros = clf_ros.predict_proba(X_test)
pred_ros = np.argmax(proba_ros, axis=1)
m_ros = evaluate_and_report(y_test, pred_ros, proba_ros, list(le.classes_))
results.append({"variant": "Random Oversample", **m_ros})

## Variant D — **SMOTE**
Synthetic minority oversampling in TF‑IDF space can work, but may be slow/high‑dimensional. Proceed with caution.
We cap features to 30k here to keep runtime reasonable. You can skip if it runs too slow.

In [ ]:
tfidf_smote = TfidfVectorizer(stop_words='english', max_df=0.7, min_df=5, max_features=30000)
Xtr_sm = tfidf_smote.fit_transform(X_train_texts)
Xte_sm = tfidf_smote.transform(X_test_texts)

try:
    sm = SMOTE(random_state=RANDOM_STATE, n_jobs=1)
    X_sm, y_sm = sm.fit_resample(Xtr_sm, y_train)
    clf_sm = xgb.XGBClassifier(
        objective='multi:softprob', num_class=num_classes, n_estimators=300,
        learning_rate=0.2, max_depth=7, subsample=0.8, colsample_bytree=0.8,
        eval_metric='mlogloss', tree_method='hist', random_state=RANDOM_STATE
    )
    clf_sm.fit(X_sm, y_sm)
    proba_sm = clf_sm.predict_proba(Xte_sm)
    pred_sm = np.argmax(proba_sm, axis=1)
    m_sm = evaluate_and_report(y_test, pred_sm, proba_sm, list(le.classes_))
    results.append({"variant": "SMOTE", **m_sm})
except Exception as e:
    print("SMOTE variant skipped due to error:", e)

## Variant E — **Chi2 Feature Selection**
Keep the top‐K discriminative terms before training XGBoost.

In [ ]:
tfidf_fs = TfidfVectorizer(stop_words='english', max_df=0.7, min_df=2, max_features=200000)
Xtr_fs = tfidf_fs.fit_transform(X_train_texts)
Xte_fs = tfidf_fs.transform(X_test_texts)

selector = SelectKBest(chi2, k=min(80000, Xtr_fs.shape[1]))
Xtr_sel = selector.fit_transform(Xtr_fs, y_train)
Xte_sel = selector.transform(Xte_fs)

clf_fs = xgb.XGBClassifier(
    objective='multi:softprob', num_class=num_classes, n_estimators=400,
    learning_rate=0.15, max_depth=8, subsample=0.8, colsample_bytree=0.8,
    eval_metric='mlogloss', tree_method='hist', random_state=RANDOM_STATE
)
clf_fs.fit(Xtr_sel, y_train)
proba_fs = clf_fs.predict_proba(Xte_sel)
pred_fs = np.argmax(proba_fs, axis=1)
m_fs = evaluate_and_report(y_test, pred_fs, proba_fs, list(le.classes_))
results.append({"variant": "Chi2 SelectKBest", **m_fs})

## Variant F — **TruncatedSVD (LSA)**
Reduce TF‑IDF to a dense latent space (e.g., 300–600 dims), then train XGBoost on reduced features.

In [ ]:
tfidf_svd = TfidfVectorizer(stop_words='english', max_df=0.7, min_df=2, max_features=200000)
Xtr_svd = tfidf_svd.fit_transform(X_train_texts)
Xte_svd = tfidf_svd.transform(X_test_texts)

svd = TruncatedSVD(n_components=400, random_state=RANDOM_STATE)
norm = Normalizer(copy=False)
Xtr_red = norm.fit_transform(svd.fit_transform(Xtr_svd))
Xte_red = norm.transform(svd.transform(Xte_svd))

clf_svd = xgb.XGBClassifier(
    objective='multi:softprob', num_class=num_classes, n_estimators=600,
    learning_rate=0.1, max_depth=8, subsample=0.9, colsample_bytree=0.9,
    eval_metric='mlogloss', tree_method='hist', random_state=RANDOM_STATE
)
clf_svd.fit(Xtr_red, y_train)
proba_svd = clf_svd.predict_proba(Xte_red)
pred_svd = np.argmax(proba_svd, axis=1)
m_svd = evaluate_and_report(y_test, pred_svd, proba_svd, list(le.classes_))
results.append({"variant": "TFIDF→SVD", **m_svd})

## Compare Variants
We compile a DataFrame of all results.

In [ ]:
df_results = pd.DataFrame(results)
df_results = df_results.sort_values(by=["macro_f1", "weighted_f1"], ascending=False)
df_results.reset_index(drop=True, inplace=True)
df_results

## Optional: Quick CV on Best Pipeline
Use a stratified 3‑fold CV to sanity‑check the best approach. (This is light‑weight, not an exhaustive search.)

In [ ]:
best_variant = df_results.iloc[0]["variant"] if not df_results.empty else "Baseline TFIDF+XGB"
print("Best variant from holdout:", best_variant)

if best_variant == "Baseline TFIDF+XGB":
    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(stop_words='english', max_df=0.7, min_df=2, max_features=50000)),
        ("xgb", xgb.XGBClassifier(
            objective='multi:softprob', num_class=num_classes, n_estimators=400,
            learning_rate=0.15, max_depth=8, subsample=0.8, colsample_bytree=0.8,
            eval_metric='mlogloss', tree_method='hist', random_state=RANDOM_STATE
        ))
    ])
elif best_variant == "Class‑Weighted":
    # Here we demonstrate a simple class‑weighting via sample_weight inside a custom CV loop
    pipe = None
else:
    pipe = None

if pipe is not None:
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    f1s = []
    for tr_idx, te_idx in skf.split(texts, y):
        Xtr_txt = [texts[i] for i in tr_idx]
        Xte_txt = [texts[i] for i in te_idx]
        ytr = y[tr_idx]
        yte = y[te_idx]
        pipe.fit(Xtr_txt, ytr)
        yhat = pipe.named_steps['xgb'].predict(pipe.named_steps['tfidf'].transform(Xte_txt))
        f1s.append(f1_score(yte, yhat, average='macro'))
    print("3‑fold Macro‑F1:", f1s, "→ mean=", np.mean(f1s))
else:
    print("Skipping quick CV for non‑pipeline variant.")

## Inference Helper (Production‑style)
This cell illustrates how to **load artifacts and predict** on raw text, returning top‑k labels with probabilities.

In [ ]:
def load_artifacts(vec_path="tfidf_vectorizer.joblib", model_path="xgb_model.joblib", label_encoder=le):
    vec = joblib.load(vec_path)
    model = joblib.load(model_path)
    return vec, model, label_encoder

def predict_texts(text_list, vec, model, label_encoder, top_k=3):
    X = vec.transform(text_list)
    proba = model.predict_proba(X)
    topk_idx = np.argsort(-proba, axis=1)[:, :top_k]
    results = []
    for i, idxs in enumerate(topk_idx):
        preds = [(label_encoder.classes_[j], float(proba[i, j])) for j in idxs]
        results.append(preds)
    return results

vec, mdl, le_used = load_artifacts()
predict_texts(["Space shuttle launches and NASA missions."], vec, mdl, le_used, top_k=3)

### Notes
- Increase `max_features` in TF‑IDF if you have ample RAM/GPU; otherwise start with 20k–50k.
- For very imbalanced data, **class‑weighted training** or **oversampling** usually boosts **macro‑F1**.
- **TruncatedSVD** can improve **speed** and sometimes **generalization**; try 300–800 components.
- Consider **early stopping** by providing an eval set to `fit()` and using `early_stopping_rounds`.